In [18]:
import sys
!{sys.executable} -m pip install scikit-learn==1.3.2 tensorflow==2.15.0 numpy


[notice] A new release of pip available: 22.2.2 -> 24.2
[notice] To update, run: pip install --upgrade pip


In [21]:

import csv
import numpy
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras

ModuleNotFoundError: No module named 'numpy.rec'

In [ ]:
with open('file1.csv', 'r') as f:
    reader = csv.reader(f)
    dataset1 = list(reader)
data_array = np.array(dataset1)

In [6]:
data_array.shape

(127, 1)

In [ ]:

scaler = MinMaxScaler(feature_range=(0, 1))
scaler = scaler.fit(dataset1)
dataset = scaler.transform(dataset1)

In [ ]:
# generate the input and output sequences
n_lookback = 60  # length of input sequences (lookback period)
n_forecast = 30  # length of output sequences (forecast period)

X = []
Y = []

for i in range(n_lookback, len(dataset) - n_forecast + 1):
    X.append(dataset[i - n_lookback: i])
    Y.append(dataset[i: i + n_forecast])

In [ ]:
X = np.array(X)
Y = np.array(Y)


In [ ]:
# fit the model
model = Sequential(name="forecast")
model.add(LSTM(units=50, return_sequences=True, input_shape=(n_lookback, 1)))
model.add(LSTM(units=50))
model.add(Dense(n_forecast))

In [ ]:

model.compile(loss='mean_squared_error', optimizer='adam')
model.fit(X, Y, epochs=100, batch_size=32, verbose=0)

In [ ]:
# generate the forecasts
X_ = dataset[- n_lookback:]  # last available input sequence
X_ = X_.reshape(1, n_lookback, 1)

Y_ = model.predict(X_).reshape(-1, 1)
Y_ = scaler.inverse_transform(Y_)

In [ ]:
# organize the results in a data frame
#df_past = df[['Close']].reset_index()
#df_past.rename(columns={'index': 'Date', 'Close': 'Actual'}, inplace=True)
#df_past['Date'] = pd.to_datetime(df_past['Date'])
#df_past['Forecast'] = np.nan
#df_past['Forecast'].iloc[-1] = df_past['Actual'].iloc[-1]

In [ ]:

#df_future = pd.DataFrame(columns=['Date', 'Actual', 'Forecast'])
#df_future['Date'] = pd.date_range(start=df_past['Date'].iloc[-1] + pd.Timedelta(days=1), periods=n_forecast)
#df_future['Forecast'] = Y_.flatten()
#df_future['Actual'] = np.nan

In [ ]:
#results = pd.concat([df_past, df_future])
#results = results.set_index('Date')

# plot the results
#results.plot(title='IBM')


In [ ]:
import os
model.save("./forecast.keras")

In [ ]:
onnx_model, _ = tf2onnx.convert.from_keras(model)
onnx.save(onnx_model, "./forecast.onnx")

In [ ]:
client = Minio(
    "minio.stock-predict.svc.cluster.local:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)


In [ ]:
buckets = client.list_buckets()
for bucket in buckets:
    print(bucket.name, bucket.creation_date)

In [ ]:
bucket_name = "models"
source_file = "./forecast.onnx"
destination_file = "forecast.onnx"
client.fput_object(bucket_name, destination_file, source_file)